# 18 — Reading Data from API Sources in PySpark

**What you will learn:**
- Why Spark cannot call APIs directly and what the correct pattern is
- Single API call → Spark DataFrame
- Paginated API → collect all pages → DataFrame
- Flattening nested JSON responses
- Parallel API calls using `ThreadPoolExecutor`
- Handling auth headers, rate limits, and retries
- Incremental load pattern (fetch only new records)
- Writing API data to Delta Lake via Unity Catalog Volumes

> **APIs used in this notebook (free, no key required):**
> - `https://jsonplaceholder.typicode.com` — fake REST API for practice
> - `https://restcountries.com` — real country data, open access

---
## Concept — Why Spark Cannot Call APIs Directly

Spark is a **distributed** engine — your code runs on many worker nodes in parallel.  
An API call inside a `map()` or `udf()` would fire from every worker independently,  
causing thousands of uncontrolled requests with no error handling.

**Correct pattern:**
```
Driver node
  │
  ├─ 1. Call API with requests / httpx  ← Python on the driver
  ├─ 2. Collect all pages into a Python list
  ├─ 3. spark.createDataFrame(list)     ← hand off to Spark
  └─ 4. Transform / write with Spark    ← distributed from here
```

Exception: when you need to **enrich each row** by calling an API per row,  
use `ThreadPoolExecutor` on the driver (controlled parallelism) — shown in Part 5.

---
## Setup

In [ ]:
import requests
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, flatten, to_timestamp, lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    BooleanType, ArrayType, DoubleType
)

CATALOG  = "main"
SCHEMA   = "default"
VOLUME   = "practice_vol"
BASE     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Base URLs
PLACEHOLDER_URL  = "https://jsonplaceholder.typicode.com"
COUNTRIES_URL    = "https://restcountries.com/v3.1"

print("Setup complete.")

---
## Part 1 — Single API Call → Spark DataFrame

In [ ]:
# ── Fetch all users from JSONPlaceholder ─────────────────────────────────────
response = requests.get(f"{PLACEHOLDER_URL}/users", timeout=10)
response.raise_for_status()          # raises HTTPError for 4xx / 5xx responses

users_raw = response.json()          # list of dicts
print(f"Status : {response.status_code}")
print(f"Records: {len(users_raw)}")
print(json.dumps(users_raw[0], indent=2))   # inspect one record

In [ ]:
# ── Convert to Spark DataFrame ────────────────────────────────────────────────
# spark.createDataFrame works directly on a list of dicts
# Spark infers the schema from the first few rows

df_users = spark.createDataFrame(users_raw)
df_users.printSchema()
df_users.show(5, truncate=False)

In [ ]:
# ── Better: define schema explicitly ─────────────────────────────────────────
# Avoids schema inference scan and makes types predictable

address_schema = StructType([
    StructField("street",  StringType(), True),
    StructField("suite",   StringType(), True),
    StructField("city",    StringType(), True),
    StructField("zipcode", StringType(), True),
])

user_schema = StructType([
    StructField("id",       IntegerType(), True),
    StructField("name",     StringType(),  True),
    StructField("username", StringType(),  True),
    StructField("email",    StringType(),  True),
    StructField("phone",    StringType(),  True),
    StructField("website",  StringType(),  True),
    StructField("address",  address_schema, True),
])

df_users_typed = spark.createDataFrame(users_raw, schema=user_schema)
df_users_typed.printSchema()
df_users_typed.show(5, truncate=False)

In [ ]:
# ── Flatten nested struct columns ─────────────────────────────────────────────
# address is a struct — extract sub-fields as top-level columns

df_users_flat = df_users_typed.select(
    col("id"),
    col("name"),
    col("email"),
    col("address.city").alias("city"),
    col("address.zipcode").alias("zipcode"),
    col("website"),
)

df_users_flat.show(truncate=False)

---
## Part 2 — Paginated API → Collect All Pages → DataFrame

Most real APIs return data in pages (`page=1`, `page=2`, ...) or use cursor-based pagination.  
You must fetch all pages on the driver before creating a DataFrame.

In [ ]:
# ── Page-number based pagination ──────────────────────────────────────────────
# JSONPlaceholder /posts has 100 records; simulate pagination with _limit & _start

def fetch_all_pages(base_url, endpoint, page_size=10):
    """Fetch all pages from a paginated REST API."""
    all_records = []
    page = 1

    while True:
        params   = {"_limit": page_size, "_page": page}
        response = requests.get(f"{base_url}{endpoint}", params=params, timeout=10)
        response.raise_for_status()

        batch = response.json()
        if not batch:           # empty page = we've fetched everything
            break

        all_records.extend(batch)
        print(f"  Page {page}: {len(batch)} records (total so far: {len(all_records)})")
        page += 1

        time.sleep(0.1)         # polite delay between requests

    return all_records


print("Fetching posts...")
posts_raw = fetch_all_pages(PLACEHOLDER_URL, "/posts", page_size=20)
print(f"Total posts fetched: {len(posts_raw)}")

In [ ]:
# ── Convert paginated result to DataFrame ─────────────────────────────────────
df_posts = spark.createDataFrame(posts_raw)
print(f"DataFrame rows: {df_posts.count()}")
df_posts.show(5, truncate=True)

In [ ]:
# ── Cursor / next-URL based pagination (common in modern APIs) ─────────────────
# Pattern: response contains a "next" URL; loop until next is None

def fetch_cursor_paginated(first_url, next_key="next"):
    """Follow 'next' links until exhausted."""
    all_records = []
    url = first_url

    while url:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data     = response.json()

        results  = data.get("results", data)    # some APIs wrap in {results: [...]}
        if isinstance(results, list):
            all_records.extend(results)

        url = data.get(next_key)                # None when last page
        time.sleep(0.1)

    return all_records

# Example call (swap in any cursor-paginated API URL):
# records = fetch_cursor_paginated("https://api.example.com/data?page_size=100")
print("Cursor pagination helper defined.")

---
## Part 3 — Deeply Nested JSON (Real API Response)

Real APIs often return deeply nested objects and arrays.  
REST Countries API returns rich nested data — good for practising flattening.

In [ ]:
# ── Fetch all countries ───────────────────────────────────────────────────────
response  = requests.get(f"{COUNTRIES_URL}/all?fields=name,capital,region,subregion,population,area,languages,currencies", timeout=15)
response.raise_for_status()
countries_raw = response.json()

print(f"Countries fetched: {len(countries_raw)}")
print(json.dumps(countries_raw[0], indent=2))

In [ ]:
# ── Normalize nested fields before creating DataFrame ─────────────────────────
# REST Countries returns nested dicts for name, currencies, languages
# Flatten to simple dicts before handing to Spark

def normalize_country(c):
    return {
        "common_name":  c.get("name", {}).get("common",  ""),
        "official_name":c.get("name", {}).get("official",""),
        "capital":      c.get("capital",  [""])[0] if c.get("capital") else None,
        "region":       c.get("region",   ""),
        "subregion":    c.get("subregion",""),
        "population":   c.get("population", 0),
        "area":         c.get("area", 0.0),
        "languages":    ", ".join(c.get("languages", {}).values()),
        "currencies":   ", ".join(c.get("currencies", {}).keys()),
    }

countries_normalized = [normalize_country(c) for c in countries_raw]

df_countries = spark.createDataFrame(countries_normalized)
df_countries.printSchema()
df_countries.show(10, truncate=False)

In [ ]:
# ── Process: top 10 most populous countries ───────────────────────────────────
from pyspark.sql.functions import desc, format_number

(
    df_countries
    .filter(col("population") > 0)
    .orderBy(desc("population"))
    .select(
        "common_name",
        "region",
        format_number("population", 0).alias("population"),
        "capital"
    )
    .limit(10)
    .show(truncate=False)
)

In [ ]:
# ── Aggregate by region ───────────────────────────────────────────────────────
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

(
    df_countries
    .groupBy("region")
    .agg(
        count("*").alias("country_count"),
        _round(_sum("population") / 1e6, 1).alias("total_population_M"),
        _round(avg("area"), 0).alias("avg_area_km2")
    )
    .orderBy(desc("total_population_M"))
    .show(truncate=False)
)

---
## Part 4 — Authentication Patterns

Most production APIs require authentication. These are the patterns you will encounter.

In [ ]:
# ── Pattern 1: Bearer Token (most common — OAuth2, JWT) ──────────────────────
# Store the token in a Databricks Secret, never hardcode it

# token = dbutils.secrets.get(scope="my-scope", key="api-token")
# headers = {"Authorization": f"Bearer {token}"}
# response = requests.get("https://api.example.com/data", headers=headers, timeout=10)

print("Bearer token pattern: store token in Databricks Secret Scope, fetch with dbutils.secrets.get")

In [ ]:
# ── Pattern 2: API Key in header or query param ───────────────────────────────

# api_key = dbutils.secrets.get(scope="my-scope", key="weather-api-key")

# # As a header (e.g. OpenWeatherMap)
# headers  = {"x-api-key": api_key}
# response = requests.get("https://api.openweathermap.org/data/2.5/weather",
#                         headers=headers, params={"q": "London"}, timeout=10)

# # As a query parameter
# response = requests.get("https://api.example.com/data",
#                         params={"apikey": api_key, "format": "json"}, timeout=10)

print("API key pattern: pass as header or query param — value always from Databricks Secrets")

In [ ]:
# ── Pattern 3: OAuth2 Client Credentials (service-to-service) ─────────────────

def get_oauth2_token(token_url, client_id, client_secret, scope=""):
    """Exchange client credentials for a Bearer token."""
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         scope,
    }
    resp = requests.post(token_url, data=payload, timeout=10)
    resp.raise_for_status()
    return resp.json()["access_token"]

# Usage:
# client_id     = dbutils.secrets.get("my-scope", "client-id")
# client_secret = dbutils.secrets.get("my-scope", "client-secret")
# token = get_oauth2_token(
#     "https://login.microsoftonline.com/<tenant>/oauth2/v2.0/token",
#     client_id, client_secret, scope="https://graph.microsoft.com/.default"
# )
# headers  = {"Authorization": f"Bearer {token}"}

print("OAuth2 helper defined.")

---
## Part 5 — Parallel API Calls with ThreadPoolExecutor

When you need to call the same API for many IDs (e.g. enrich 500 user IDs),  
sequential calls are too slow. Use `ThreadPoolExecutor` on the driver for  
controlled parallel HTTP requests.

In [ ]:
# ── Fetch posts for multiple user IDs in parallel ─────────────────────────────

def fetch_posts_for_user(user_id):
    """Return (user_id, list_of_posts) — called from worker threads."""
    try:
        resp = requests.get(
            f"{PLACEHOLDER_URL}/posts",
            params={"userId": user_id},
            timeout=10
        )
        resp.raise_for_status()
        return user_id, resp.json()
    except requests.RequestException as e:
        print(f"  ERROR user_id={user_id}: {e}")
        return user_id, []


user_ids    = list(range(1, 11))   # user IDs 1–10
all_results = []

# max_workers controls concurrency — keep low to avoid rate-limiting
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(fetch_posts_for_user, uid): uid for uid in user_ids}

    for future in as_completed(futures):
        uid, posts = future.result()
        all_results.extend(posts)
        print(f"  user_id={uid}: {len(posts)} posts")

print(f"\nTotal posts collected: {len(all_results)}")

In [ ]:
# ── Convert parallel results to DataFrame and process ─────────────────────────
df_all_posts = spark.createDataFrame(all_results)

# Count posts per user
(
    df_all_posts
    .groupBy("userId")
    .count()
    .orderBy("userId")
    .show()
)

---
## Part 6 — Retry Logic and Error Handling

In [ ]:
# ── Robust fetch with exponential backoff retry ───────────────────────────────

def fetch_with_retry(url, params=None, headers=None, max_retries=3, backoff=2):
    """
    GET request with exponential backoff.
    Retries on 429 (rate limit) and 5xx (server errors).
    """
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params, headers=headers, timeout=10)

            if resp.status_code == 429:                  # rate limited
                wait = int(resp.headers.get("Retry-After", backoff ** attempt))
                print(f"  Rate limited. Waiting {wait}s before retry {attempt}...")
                time.sleep(wait)
                continue

            if resp.status_code >= 500:                  # server error — retry
                print(f"  Server error {resp.status_code}. Retry {attempt}/{max_retries}...")
                time.sleep(backoff ** attempt)
                continue

            resp.raise_for_status()                      # raise on 4xx
            return resp.json()

        except requests.ConnectionError:
            print(f"  Connection error. Retry {attempt}/{max_retries}...")
            time.sleep(backoff ** attempt)

    raise RuntimeError(f"All {max_retries} retries exhausted for {url}")


# Usage
data = fetch_with_retry(f"{PLACEHOLDER_URL}/todos", params={"userId": 1})
print(f"Fetched {len(data)} todos with retry-safe function")

---
## Part 7 — Incremental Load Pattern

In production pipelines you don't reload all data every run.  
You fetch only records created/modified since the last successful run.

In [ ]:
# ── Watermark helpers ─────────────────────────────────────────────────────────
import os

WATERMARK_PATH = f"{BASE}/api_watermark.txt"

def read_watermark(default="1970-01-01T00:00:00Z"):
    """Read last successful run timestamp from the volume."""
    try:
        content = dbutils.fs.head(WATERMARK_PATH, 64)
        return content.strip()
    except Exception:
        return default

def write_watermark(value):
    """Persist the new high-water mark after a successful run."""
    dbutils.fs.put(WATERMARK_PATH, value, overwrite=True)


last_run = read_watermark()
print(f"Last successful run: {last_run}")

In [ ]:
# ── Simulated incremental fetch ───────────────────────────────────────────────
# JSONPlaceholder doesn't support since= filtering; in real APIs you'd pass:
#   params={"updated_since": last_run, "page_size": 500}

from datetime import datetime, timezone

def incremental_fetch(since_timestamp):
    """Fetch records updated after since_timestamp (ISO 8601 string)."""
    # In a real API this would be a query param:
    # params = {"updated_since": since_timestamp, "limit": 1000}
    # For demo purposes, fetch all and filter in Python
    records = fetch_with_retry(f"{PLACEHOLDER_URL}/comments")
    # Simulate: keep only records with id > some threshold derived from timestamp
    new_records = [r for r in records if r["id"] > 450]
    return new_records


new_data = incremental_fetch(last_run)
print(f"New records since {last_run}: {len(new_data)}")

if new_data:
    df_new = spark.createDataFrame(new_data).withColumn("ingested_at", current_timestamp())
    df_new.show(5)

    # Append to Delta table
    df_new.write.format("delta").mode("append").save(f"{BASE}/api_comments_delta/")

    # Update watermark only after successful write
    new_watermark = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    write_watermark(new_watermark)
    print(f"Watermark updated to: {new_watermark}")
else:
    print("No new records — skipping write.")

---
## Part 8 — Write API Data to Delta Lake

In [ ]:
# ── Write countries to a managed Delta table ──────────────────────────────────
(
    df_countries
    .withColumn("ingested_at", current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.api_countries")
)

print("Written to Delta table: main.default.api_countries")

In [ ]:
# ── Write posts with MERGE (upsert) — avoid duplicates on re-runs ─────────────
from delta.tables import DeltaTable

posts_path = f"{BASE}/api_posts_delta/"

df_posts_staged = (
    df_all_posts
    .withColumn("ingested_at", current_timestamp())
)

if DeltaTable.isDeltaTable(spark, posts_path):
    # Table exists — upsert on primary key
    delta_table = DeltaTable.forPath(spark, posts_path)
    (
        delta_table.alias("target")
        .merge(
            df_posts_staged.alias("source"),
            "target.id = source.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Upsert complete.")
else:
    # First run — create table
    df_posts_staged.write.format("delta").mode("overwrite").save(posts_path)
    print("Initial load complete.")

spark.read.format("delta").load(posts_path).show(5)

---
## Key Takeaways

| Pattern | When to use |
|---|---|
| `requests.get` → `spark.createDataFrame` | Single endpoint, all data fits in driver memory |
| Pagination loop → list → DataFrame | API returns data across multiple pages |
| Normalize before `createDataFrame` | Deeply nested JSON — flatten on driver, not in Spark |
| `ThreadPoolExecutor` | Fetch same API for many IDs — controlled parallel calls |
| Retry with exponential backoff | Production pipelines — handle 429 / 5xx gracefully |
| Watermark file in Volume | Incremental loads — avoid re-fetching all data every run |
| Delta MERGE (upsert) | Re-runnable pipelines — no duplicate rows on retry |
| `dbutils.secrets.get` | API keys / tokens — never hardcode credentials |

### Interview one-liners
- **Why not call APIs inside a Spark UDF?** Uncontrolled fan-out — one UDF call per row across all workers with no rate limiting or error handling
- **How do you handle pagination?** Loop on driver until empty page or no `next` cursor, accumulate into a list, then `createDataFrame` once
- **How do you avoid re-processing data on retries?** Watermark stored in a Volume or Delta table; fetch only records after the last watermark
- **Where do you store API credentials?** Databricks Secret Scope — `dbutils.secrets.get(scope, key)` — values are always redacted in output